In [2]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import numpy as np
import jax
import jax.numpy as jnp
from jax import random, vmap
import matplotlib.pyplot as plt
import arviz as az
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from pathlib import Path
from datetime import datetime
import time

from ThreeNeuron_JAX import (
    ModelParams,
    NoiseParams,
    simulate_pair_jit,
    pred_sigma,
)

jax.config.update("jax_enable_x64", True)

# ======================================================
# Local paths 
# ======================================================
BASE_DIR = Path(".") / "mcmc_3neuron2"
DATA_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "outputs"

# create directories if they don't exist
BASE_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

# ======================================================
# Experiment settings
# ======================================================
W_LOW, W_HIGH = 1.5, 9.5
FIXED_WEIGHT_PAIRS = [(4.0, 5.5), (4.5, 7.0), (6.0, 6.0), (8.0, 7.5)]
CASES_PER_WEIGHT = 1
K = 1

WARMUP, DRAWS = 4000, 4000
CHAINS = 4
TARGET_ACCEPT = 0.9
GRID_N = 81

cfg = ModelParams()
noise = NoiseParams(sigma_f=4e-4, gamma=1e-4, sigma_c=28.0, floor=1e-3)

# ======================================================
# Simulator
# ======================================================
@jax.jit
def forward_CF(w_vec):
    """
    Run the 3-neuron simulator for given weights
    returns: (C, F) of shape (3, T)
    """
    return simulate_pair_jit(jnp.asarray(w_vec, dtype=jnp.float64), cfg)

def fwd_sigma_for_K(w_vec, K_obs):
    """
    Broadcast C,F and compute sigma for K observations
    """
    C2, F2 = forward_CF(w_vec)
    Ck = jnp.broadcast_to(C2, (K_obs, *C2.shape))
    Fk = jnp.broadcast_to(F2, (K_obs, *F2.shape))
    Sig = pred_sigma(Fk, Ck, cfg, noise)
    return Fk, Sig

# ======================================================
# Pooled NLL grid
# ======================================================
def pooled_nll_grid(Y_obs, w0_grid, w1_grid):
    K_obs = Y_obs.shape[0]

    def nll_at(w_vec):
        Fk, Sig = fwd_sigma_for_K(w_vec, K_obs)
        z = (Fk - Y_obs) / Sig
        return jnp.sum(0.5 * (z*z + 2.0*jnp.log(Sig) + jnp.log(2.0*jnp.pi)))

    W0, W1 = jnp.meshgrid(w0_grid, w1_grid, indexing="ij")
    W = jnp.stack([W0, W1], axis=-1)
    return vmap(lambda row: vmap(nll_at)(row))(W)

# ======================================================
# NumPyro model
# ======================================================
def model(Y_obs):
    K_obs = Y_obs.shape[0]
    w0 = numpyro.sample("w0", dist.Uniform(W_LOW, W_HIGH))
    w1 = numpyro.sample("w1", dist.Uniform(W_LOW, W_HIGH))
    Fk, Sig = fwd_sigma_for_K(jnp.array([w0, w1]), K_obs)

    numpyro.sample(
        "y",
        dist.Normal(Fk.reshape(K_obs, -1), Sig.reshape(K_obs, -1)).to_event(1),
        obs=Y_obs.reshape(K_obs, -1)
    )

def make_mcmc():
    kernel = NUTS(model, target_accept_prob=TARGET_ACCEPT, max_tree_depth=8)
    return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
                num_chains=CHAINS, chain_method="parallel", progress_bar=True)

def run_mcmc(Y_obs, seed=0):
    mcmc = make_mcmc()
    mcmc.run(random.PRNGKey(seed), Y_obs=jnp.asarray(Y_obs, dtype=jnp.float64))
    return az.from_numpyro(mcmc)

# ======================================================
# Load observations
# ======================================================
def get_Y_obs(w0, w1, K, rng):
    fname = DATA_DIR / f"Y_w0_{w0}_w1_{w1}.npy"
    Y = np.load(fname)
    idx = rng.integers(0, Y.shape[0], size=K)
    return Y[idx].astype(np.float64)

# ======================================================
# Main
# ======================================================
if __name__ == "__main__":
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    runtimes = []

    for i, (w0_true, w1_true) in enumerate(FIXED_WEIGHT_PAIRS):
        rng = np.random.default_rng(1000 + i)
        Y_obs = get_Y_obs(w0_true, w1_true, K, rng)

        tag = f"w0_{w0_true}_w1_{w1_true}_K{K}_{ts}"
        out_dir = OUT_DIR / tag
        out_dir.mkdir(parents=True, exist_ok=True)

        # ----- NLL grid -----
        w0_grid = jnp.linspace(W_LOW, W_HIGH, GRID_N)
        w1_grid = jnp.linspace(W_LOW, W_HIGH, GRID_N)
        nll_map = pooled_nll_grid(Y_obs, w0_grid, w1_grid)

        plt.figure(figsize=(6,5))
        plt.imshow(np.asarray(nll_map), origin="lower",
                   extent=[W_LOW,W_HIGH,W_LOW,W_HIGH], aspect="auto")
        plt.scatter([w0_true], [w1_true], c="r", s=40)
        plt.colorbar(label="Pooled NLL")
        plt.xlabel("w0")
        plt.ylabel("w1")
        plt.tight_layout()
        plt.savefig(out_dir / "nll_grid.png", dpi=160)
        plt.close()

        # ----- MCMC -----
        t0 = time.perf_counter()
        idata = run_mcmc(Y_obs, seed=2025+i)
        runtimes.append(time.perf_counter() - t0)

        az.to_netcdf(idata, out_dir / "posterior.nc")

        az.plot_trace(idata, var_names=["w0","w1"])
        plt.tight_layout()
        plt.savefig(out_dir / "trace.png", dpi=160)
        plt.close()

        az.plot_posterior(idata, var_names=["w0","w1"])
        plt.tight_layout()
        plt.savefig(out_dir / "posterior.png", dpi=160)
        plt.close()
        
    print("Average runtime per case (s):", np.mean(runtimes))


/tmp/ipykernel_147664/3989442761.py:108: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|██████████| 8000/8000 [02:44<00:00, 48.60it/s, 119 steps of size 2.20e-02. acc. prob=0.95]
/tmp/ipykernel_147664/3989442761.py:108: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|█

Average runtime per case (s): 378.37526952358894


In [1]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import numpy as np
import jax
import jax.numpy as jnp
from jax import random, vmap
import matplotlib.pyplot as plt
import arviz as az
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from pathlib import Path
from datetime import datetime
import time

from ThreeNeuron_JAX import (
    ModelParams,
    NoiseParams,
    simulate_pair_jit,
    pred_sigma,
)

jax.config.update("jax_enable_x64", True)

# ======================================================
# Local paths 
# ======================================================
BASE_DIR = Path(".") / "mcmc_3neuron2"
DATA_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "outputs2"

# create directories if they don't exist
BASE_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

# ======================================================
# Experiment settings
# ======================================================
W_LOW, W_HIGH = 1.5, 9.5
FIXED_WEIGHT_PAIRS = [(4.0, 5.5), (4.5, 7.0), (6.0, 6.0), (8.0, 7.5)]
CASES_PER_WEIGHT = 5
K = 3

WARMUP, DRAWS = 4000, 4000
CHAINS = 4
TARGET_ACCEPT = 0.9
GRID_N = 81

cfg = ModelParams()
noise = NoiseParams(sigma_f=8e-3, gamma=1e-4, sigma_c=28.0, floor=1e-3)

# ======================================================
# Simulator
# ======================================================
@jax.jit
def forward_CF(w_vec):
    """
    Run the 3-neuron simulator for given weights
    returns: (C, F) of shape (3, T)
    """
    return simulate_pair_jit(jnp.asarray(w_vec, dtype=jnp.float64), cfg)

def fwd_sigma_for_K(w_vec, K_obs):
    """
    Broadcast C,F and compute sigma for K observations
    """
    C2, F2 = forward_CF(w_vec)
    Ck = jnp.broadcast_to(C2, (K_obs, *C2.shape))
    Fk = jnp.broadcast_to(F2, (K_obs, *F2.shape))
    Sig = pred_sigma(Fk, Ck, cfg, noise)
    return Fk, Sig

# ======================================================
# Pooled NLL grid
# ======================================================
def pooled_nll_grid(Y_obs, w0_grid, w1_grid):
    K_obs = Y_obs.shape[0]

    def nll_at(w_vec):
        Fk, Sig = fwd_sigma_for_K(w_vec, K_obs)
        z = (Fk - Y_obs) / Sig
        return jnp.sum(0.5 * (z*z + 2.0*jnp.log(Sig) + jnp.log(2.0*jnp.pi)))

    W0, W1 = jnp.meshgrid(w0_grid, w1_grid, indexing="ij")
    W = jnp.stack([W0, W1], axis=-1)
    return vmap(lambda row: vmap(nll_at)(row))(W)

# ======================================================
# NumPyro model
# ======================================================
def model(Y_obs):
    K_obs = Y_obs.shape[0]
    w0 = numpyro.sample("w0", dist.Uniform(W_LOW, W_HIGH))
    w1 = numpyro.sample("w1", dist.Uniform(W_LOW, W_HIGH))
    Fk, Sig = fwd_sigma_for_K(jnp.array([w0, w1]), K_obs)

    numpyro.sample(
        "y",
        dist.Normal(Fk.reshape(K_obs, -1), Sig.reshape(K_obs, -1)).to_event(1),
        obs=Y_obs.reshape(K_obs, -1)
    )

def make_mcmc():
    kernel = NUTS(model, target_accept_prob=TARGET_ACCEPT, max_tree_depth=8)
    return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
                num_chains=CHAINS, chain_method="parallel", progress_bar=True)

def run_mcmc(Y_obs, seed=0):
    mcmc = make_mcmc()
    mcmc.run(random.PRNGKey(seed), Y_obs=jnp.asarray(Y_obs, dtype=jnp.float64))
    return az.from_numpyro(mcmc)

# ======================================================
# Load observations
# ======================================================
def get_Y_obs(w0, w1, K, rng):
    fname = DATA_DIR / f"Y_w0_{w0}_w1_{w1}.npy"
    Y = np.load(fname)
    idx = rng.integers(0, Y.shape[0], size=K)
    return Y[idx].astype(np.float64)

# ======================================================
# Main
# ======================================================
if __name__ == "__main__":
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    runtimes = []

    for i, (w0_true, w1_true) in enumerate(FIXED_WEIGHT_PAIRS):

        for case in range(CASES_PER_WEIGHT):

            rng = np.random.default_rng(1000 + i*100 + case)
            Y_obs = get_Y_obs(w0_true, w1_true, K, rng)

            tag = f"w0_{w0_true}_w1_{w1_true}_K{K}_case{case}_{ts}"
            out_dir = OUT_DIR / tag
            out_dir.mkdir(parents=True, exist_ok=True)

            # ----- NLL grid -----
            w0_grid = jnp.linspace(W_LOW, W_HIGH, GRID_N)
            w1_grid = jnp.linspace(W_LOW, W_HIGH, GRID_N)
            nll_map = pooled_nll_grid(Y_obs, w0_grid, w1_grid)

            plt.figure(figsize=(6,5))
            plt.imshow(np.asarray(nll_map), origin="lower",
                       extent=[W_LOW,W_HIGH,W_LOW,W_HIGH], aspect="auto")
            plt.scatter([w0_true], [w1_true], c="r", s=40)
            plt.colorbar(label="Pooled NLL")
            plt.xlabel("w0")
            plt.ylabel("w1")
            plt.tight_layout()
            plt.savefig(out_dir / "nll_grid.png", dpi=160)
            plt.close()

            # ----- MCMC -----
            t0 = time.perf_counter()
            idata = run_mcmc(Y_obs, seed=2025 + i*100 + case)
            runtimes.append(time.perf_counter() - t0)

            # --- Compute RMSE ---
            w0_samples = idata.posterior["w0"].values.reshape(-1)
            w1_samples = idata.posterior["w1"].values.reshape(-1)
            rmse_w0 = np.sqrt(np.mean((w0_samples - w0_true)**2))
            rmse_w1 = np.sqrt(np.mean((w1_samples - w1_true)**2))
            print(f"MCMC RMSE: w0={rmse_w0:.6f}, w1={rmse_w1:.6f}")

            az.to_netcdf(idata, out_dir / "posterior.nc")

            az.plot_trace(idata, var_names=["w0","w1"])
            plt.tight_layout()
            plt.savefig(out_dir / "trace.png", dpi=160)
            plt.close()

            az.plot_posterior(idata, var_names=["w0","w1"])
            plt.tight_layout()
            plt.savefig(out_dir / "posterior.png", dpi=160)
            plt.close()

    print("Average runtime per case (s):", np.mean(runtimes))

/tmp/ipykernel_2127973/2524497793.py:108: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|██████████| 8000/8000 [03:30<00:00, 38.09it/s, 107 steps of size 1.78e-02. acc. prob=0.96]


MCMC RMSE: w0=0.103191, w1=0.127539


/tmp/ipykernel_2127973/2524497793.py:108: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|██████████| 8000/8000 [03:01<00:00, 44.16it/s, 3 steps of size 1.84e-02. acc. prob=0.96]  


MCMC RMSE: w0=0.157641, w1=0.195230


/tmp/ipykernel_2127973/2524497793.py:108: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|██████████| 8000/8000 [02:51<00:00, 46.77it/s, 3 steps of size 2.10e-02. acc. prob=0.95]  


...

MCMC RMSE: w0=0.027854, w1=0.050919


/tmp/ipykernel_2127973/2524497793.py:108: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|██████████| 8000/8000 [00:30<00:00, 261.82it/s, 3 steps of size 2.43e-01. acc. prob=0.92] 


MCMC RMSE: w0=0.042552, w1=0.060098


/tmp/ipykernel_2127973/2524497793.py:108: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|██████████| 8000/8000 [00:29<00:00, 267.60it/s, 3 steps of size 2.29e-01. acc. prob=0.92] 


MCMC RMSE: w0=0.027636, w1=0.045312
Average runtime per case (s): 421.1454560964368


In [2]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import numpy as np
import jax
import jax.numpy as jnp
from jax import random, vmap
import matplotlib.pyplot as plt
import arviz as az
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from pathlib import Path
from datetime import datetime
import time

from ThreeNeuron_JAX import (
    ModelParams,
    NoiseParams,
    simulate_pair_jit,
    pred_sigma,
)

jax.config.update("jax_enable_x64", True)

BASE_DIR = Path(".") / "mcmc_3neuron2"
DATA_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "outputs2"

BASE_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

W_LOW, W_HIGH = 1.5, 9.5
FIXED_WEIGHT_PAIRS = [(4.0, 5.5), (4.5, 7.0), (6.0, 6.0), (8.0, 7.5)]
CASES_PER_WEIGHT = 3
K = 3

WARMUP, DRAWS = 4000, 4000
CHAINS = 4
TARGET_ACCEPT = 0.9
GRID_N = 81

cfg = ModelParams()
noise = NoiseParams(sigma_f=8e-3, gamma=1e-4, sigma_c=28.0, floor=1e-3)

@jax.jit
def forward_CF(w_vec):
    return simulate_pair_jit(jnp.asarray(w_vec, dtype=jnp.float64), cfg)

def fwd_sigma_for_K(w_vec, K_obs):
    C2, F2 = forward_CF(w_vec)
    Ck = jnp.broadcast_to(C2, (K_obs, *C2.shape))
    Fk = jnp.broadcast_to(F2, (K_obs, *F2.shape))
    Sig = pred_sigma(Fk, Ck, cfg, noise)
    return Fk, Sig

def pooled_nll_grid(Y_obs, w0_grid, w1_grid):
    K_obs = Y_obs.shape[0]

    def nll_at(w_vec):
        Fk, Sig = fwd_sigma_for_K(w_vec, K_obs)
        z = (Fk - Y_obs) / Sig
        return jnp.sum(0.5 * (z*z + 2.0*jnp.log(Sig) + jnp.log(2.0*jnp.pi)))

    W0, W1 = jnp.meshgrid(w0_grid, w1_grid, indexing="ij")
    W = jnp.stack([W0, W1], axis=-1)
    return vmap(lambda row: vmap(nll_at)(row))(W)

def model(Y_obs):
    K_obs = Y_obs.shape[0]
    w0 = numpyro.sample("w0", dist.Uniform(W_LOW, W_HIGH))
    w1 = numpyro.sample("w1", dist.Uniform(W_LOW, W_HIGH))
    Fk, Sig = fwd_sigma_for_K(jnp.array([w0, w1]), K_obs)

    numpyro.sample(
        "y",
        dist.Normal(Fk.reshape(K_obs, -1), Sig.reshape(K_obs, -1)).to_event(1),
        obs=Y_obs.reshape(K_obs, -1)
    )

def make_mcmc():
    kernel = NUTS(model, target_accept_prob=TARGET_ACCEPT, max_tree_depth=8)
    return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
                num_chains=CHAINS, chain_method="parallel", progress_bar=True)

def run_mcmc(Y_obs, seed=0):
    mcmc = make_mcmc()
    mcmc.run(random.PRNGKey(seed), Y_obs=jnp.asarray(Y_obs, dtype=jnp.float64))
    return az.from_numpyro(mcmc)

def get_Y_obs(w0, w1, K, rng, out_dir):
    fname = DATA_DIR / f"Y_w0_{w0}_w1_{w1}.npy"
    Y = np.load(fname)
    idx = rng.integers(0, Y.shape[0], size=K)

    np.save(out_dir / "indices.npy", idx)

    return Y[idx].astype(np.float64)

if __name__ == "__main__":
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    runtimes = []

    for i, (w0_true, w1_true) in enumerate(FIXED_WEIGHT_PAIRS):

        for case in range(CASES_PER_WEIGHT):

            tag = f"w0_{w0_true}_w1_{w1_true}_K{K}_case{case}_{ts}"
            out_dir = OUT_DIR / tag
            out_dir.mkdir(parents=True, exist_ok=True)

            rng = np.random.default_rng(1000 + i*100 + case)
            Y_obs = get_Y_obs(w0_true, w1_true, K, rng, out_dir)

            w0_grid = jnp.linspace(W_LOW, W_HIGH, GRID_N)
            w1_grid = jnp.linspace(W_LOW, W_HIGH, GRID_N)
            nll_map = pooled_nll_grid(Y_obs, w0_grid, w1_grid)

            plt.figure(figsize=(6,5))
            plt.imshow(np.asarray(nll_map), origin="lower",
                       extent=[W_LOW,W_HIGH,W_LOW,W_HIGH], aspect="auto")
            plt.scatter([w0_true], [w1_true], c="r", s=40)
            plt.colorbar(label="Pooled NLL")
            plt.xlabel("w0")
            plt.ylabel("w1")
            plt.tight_layout()
            plt.savefig(out_dir / "nll_grid.png", dpi=160)
            plt.close()

            t0 = time.perf_counter()
            idata = run_mcmc(Y_obs, seed=2025 + i*100 + case)
            runtimes.append(time.perf_counter() - t0)

            w0_samples = idata.posterior["w0"].values.reshape(-1)
            w1_samples = idata.posterior["w1"].values.reshape(-1)
            rmse_w0 = np.sqrt(np.mean((w0_samples - w0_true)**2))
            rmse_w1 = np.sqrt(np.mean((w1_samples - w1_true)**2))
            print(f"MCMC RMSE: w0={rmse_w0:.6f}, w1={rmse_w1:.6f}")

            az.to_netcdf(idata, out_dir / "posterior.nc")

            az.plot_trace(idata, var_names=["w0","w1"])
            plt.tight_layout()
            plt.savefig(out_dir / "trace.png", dpi=160)
            plt.close()

            az.plot_posterior(idata, var_names=["w0","w1"])
            plt.tight_layout()
            plt.savefig(out_dir / "posterior.png", dpi=160)
            plt.close()

    print("Average runtime per case (s):", np.mean(runtimes))

/tmp/ipykernel_2127973/3242189803.py:85: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|██████████| 8000/8000 [03:28<00:00, 38.28it/s, 107 steps of size 1.78e-02. acc. prob=0.96]


MCMC RMSE: w0=0.103191, w1=0.127539


/tmp/ipykernel_2127973/3242189803.py:85: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|██████████| 8000/8000 [03:00<00:00, 44.23it/s, 3 steps of size 1.84e-02. acc. prob=0.96]  


MCMC RMSE: w0=0.157641, w1=0.195230


/tmp/ipykernel_2127973/3242189803.py:85: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|██████████| 8000/8000 [02:50<00:00, 47.02it/s, 3 steps of size 2.10e-02. acc. prob=0.95]  


...

MCMC RMSE: w0=0.026858, w1=0.043206


/tmp/ipykernel_2127973/3242189803.py:85: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|██████████| 8000/8000 [00:36<00:00, 221.74it/s, 7 steps of size 1.61e-01. acc. prob=0.96] 


MCMC RMSE: w0=0.027599, w1=0.045210


/tmp/ipykernel_2127973/3242189803.py:85: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  return MCMC(kernel, num_warmup=WARMUP, num_samples=DRAWS,
sample: 100%|██████████| 8000/8000 [00:36<00:00, 218.27it/s, 15 steps of size 1.60e-01. acc. prob=0.97]


MCMC RMSE: w0=0.027854, w1=0.050919
Average runtime per case (s): 415.3095544522318
